# Create Millennium Technology Prize Awards (PRIZE PATTERN)

Millennium Technology Prize winners from the official Millennium Technology Prize website.

**Prerequisite:** run `scripts/local/millennium_prize_to_s3.py` to fetch the official winners page, verify the official prize story/amount rule, fetch official detail pages, and upload parquet to S3.

**Source:** `https://millenniumprize.org/winners/` plus official winner detail pages; amount-rule source is `https://millenniumprize.org/prize/story/`.

**S3 location:** `s3://openalex-ingest/awards/millennium_prize/millennium_prize_laureates.parquet`

**OpenAlex funder:** Tekniikan Akatemia / Technology Academy Finland, `funder_id = 4320324443`, DOI `10.13039/501100007456`; no ROR currently exposed by OpenAlex.

**Schema notes:**
- Prize pattern: one row per official laureate. The 2020 prize has two co-laureates and is split into two rows.
- `lead_investigator` is the laureate.
- `funding_type = 'prize'`.
- Provenance is `millennium_prize`.
- The official prize story calls this a EUR 1 million award first awarded in 2004. `amount` is EUR 1,000,000 divided by the count of laureates for the same winning innovation.
- The source publishes country, but not a structured affiliation-when-awarded field; `lead_investigator.affiliation.name` is NULL and `affiliation.country` preserves the source country string.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.millennium_prize_raw
USING delta AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/millennium_prize/millennium_prize_laureates.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.millennium_prize_raw;


## Step 1.5: Inspect Raw Data First

Verify exact source columns and values before relying on the transform below. The source script writes all raw columns as strings with `df.astype("string")` before parquet.


In [ ]:
%sql
DESCRIBE openalex.awards.millennium_prize_raw;


In [ ]:
%sql
SELECT *
FROM openalex.awards.millennium_prize_raw
LIMIT 10;


In [ ]:
%sql
-- Money-shaped columns from the raw source. Amounts are derived in the script
-- from the official Millennium Technology Prize story and stored as strings.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'millennium_prize_raw'
  AND LOWER(column_name) RLIKE 'amount|currency|fund|award|value|money|prize|eur|euro|portion';


In [ ]:
%sql
SELECT
    currency,
    COUNT(*) AS rows,
    COUNT(source_award_amount) AS rows_with_amount,
    MIN(TRY_CAST(source_award_amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(source_award_amount AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(source_award_amount AS DOUBLE)) AS avg_amount,
    collect_set(amount_rule_url) AS amount_rule_urls
FROM openalex.awards.millennium_prize_raw
GROUP BY currency
ORDER BY currency;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(funder_award_id) AS has_funder_award_id,
    COUNT(laureate_name) AS has_name,
    COUNT(award_year) AS has_year,
    COUNT(innovation) AS has_innovation,
    COUNT(country) AS has_country,
    COUNT(profile_description) AS has_profile_description,
    COUNT(source_award_amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    MIN(TRY_CAST(award_year AS INT)) AS min_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_year
FROM openalex.awards.millennium_prize_raw;


In [ ]:
%sql
SELECT
    award_year,
    innovation,
    laureate_name,
    country,
    source_award_amount,
    currency,
    award_share_count,
    portion,
    landing_page_url
FROM openalex.awards.millennium_prize_raw
ORDER BY TRY_CAST(award_year AS INT) DESC, innovation, laureate_name;


## Step 1.6: Funder Existence Fail-Fast

Must return exactly one row. If zero rows, stop before transforming because the `CROSS JOIN` would emit zero awards.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320324443;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.millennium_prize_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320324443
),
normalized AS (
    SELECT
        n.*,
        TRY_CAST(n.award_year AS INT) AS award_year_int,
        TRY_CAST(n.source_award_amount AS DOUBLE) AS amount_double
    FROM openalex.awards.millennium_prize_raw n
),
awards AS (
    SELECT
        abs(xxhash64(CONCAT(
            TRY_CAST(f.funder_id AS STRING), ':millennium-prize:', LOWER(n.funder_award_id)
        ))) % 9000000000 AS id,
        CONCAT('Millennium Technology Prize ', TRY_CAST(n.award_year_int AS STRING), ' - ', n.innovation, ' - ', n.laureate_name) AS display_name,
        NULLIF(n.profile_description, '') AS description,
        f.funder_id,
        n.funder_award_id,
        n.amount_double AS amount,
        NULLIF(n.currency, '') AS currency,
        struct(
            CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'prize' AS funding_type,
        NULLIF(n.innovation, '') AS funder_scheme,
        'millennium_prize' AS provenance,
        TRY_TO_DATE(CONCAT(TRY_CAST(n.award_year_int AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
        TRY_TO_DATE(CONCAT(TRY_CAST(n.award_year_int AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
        n.award_year_int AS start_year,
        n.award_year_int AS end_year,
        struct(
            NULLIF(n.given_name, '') AS given_name,
            NULLIF(n.family_name, '') AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                CAST(NULL AS STRING) AS name,
                NULLIF(n.country, '') AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        NULLIF(n.landing_page_url, '') AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM normalized n
    CROSS JOIN funder_resolved f
    WHERE n.funder_award_id IS NOT NULL
      AND n.award_year_int IS NOT NULL
      AND n.laureate_name IS NOT NULL
)
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G', TRY_CAST(id AS STRING)) AS works_api_url,
    created_date,
    updated_date
FROM awards;


## Step 3: Insert Into `openalex_awards_raw` With Priority 67

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'millennium_prize' AND priority = 67;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    67 AS priority
FROM openalex.awards.millennium_prize_awards;


## Verification

In [ ]:
%sql
SELECT COUNT(*) AS total
FROM openalex.awards.millennium_prize_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.millennium_prize_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_display_name,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_lead_investigator,
    COUNT(landing_page_url) AS has_landing_page_url,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.millennium_prize_awards;


In [ ]:
%sql
SELECT
    funder.display_name AS funder_display_name,
    funder_id,
    COUNT(*) AS rows
FROM openalex.awards.millennium_prize_awards
GROUP BY funder.display_name, funder_id;


In [ ]:
%sql
SELECT
    funder_scheme,
    currency,
    COUNT(*) AS rows,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount
FROM openalex.awards.millennium_prize_awards
GROUP BY funder_scheme, currency
ORDER BY funder_scheme, currency;


In [ ]:
%sql
SELECT
    start_year,
    COUNT(*) AS rows
FROM openalex.awards.millennium_prize_awards
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
-- Duplicate checks. Both should return zero rows.
SELECT id, COUNT(*) AS rows
FROM openalex.awards.millennium_prize_awards
GROUP BY id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.millennium_prize_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT
    id,
    display_name,
    description,
    amount,
    currency,
    funder_scheme,
    start_year,
    lead_investigator,
    landing_page_url
FROM openalex.awards.millennium_prize_awards
ORDER BY start_year DESC, funder_scheme, display_name;
